In [11]:
%load_ext autoreload
%autoreload 2

In [7]:
from nnsight import LanguageModel

import gc
import itertools
import math
import os
import random
import sys
from collections import Counter
from copy import deepcopy
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Any, Callable, Literal, TypeAlias

import einops
import numpy as np
import pandas as pd
import plotly.express as px
import requests
import torch as t
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from IPython.display import HTML, IFrame, clear_output, display
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from sae_lens import (
    SAE,
    ActivationsStore,
    HookedSAETransformer,
    LanguageModelSAERunnerConfig,
    SAEConfig,
    SAETrainingRunner,
    upload_saes_to_huggingface,
)
from sae_lens.toolkit.pretrained_saes_directory import get_pretrained_saes_directory
from sae_vis import SaeVisConfig, SaeVisData, SaeVisLayoutConfig
from tabulate import tabulate
from torch import Tensor, nn
from torch.distributions.categorical import Categorical
from torch.nn import functional as F
from tqdm.auto import tqdm
from transformer_lens import ActivationCache, HookedTransformer, utils
from transformer_lens.hook_points import HookPoint

device = "cuda" if t.cuda.is_available() else "mps" if t.backends.mps.is_available() else "cpu"

sys.path.append('../scripts')
import enrichment_utils
import perturbation

In [3]:
## check memory usage
if t.cuda.is_available():
    gpu_id = 0  # Set to your target GPU ID
    total_memory = t.cuda.get_device_properties(gpu_id).total_memory
    allocated_memory = t.cuda.memory_allocated(gpu_id)
    cached_memory = t.cuda.memory_reserved(gpu_id)

    print(f"Total GPU Memory: {total_memory / 1024**2:.2f} MB")
    print(f"Allocated GPU Memory: {allocated_memory / 1024**2:.2f} MB")
    print(f"Cached GPU Memory: {cached_memory / 1024**2:.2f} MB")
elif t.backends.mps.is_available():
    # MPS (Metal Performance Shaders) for Mac
    print("MPS is available.")
    # Note: As of now, PyTorch doesn't provide direct memory management functions for MPS
    print("Memory information is not available for MPS.")
else:
    print("Neither CUDA nor MPS is available.")

Total GPU Memory: 32500.88 MB
Allocated GPU Memory: 0.00 MB
Cached GPU Memory: 0.00 MB


# Functions

In [38]:
def sae_hook(
    activations: Float[Tensor, "batch pos d_in"],
    hook: HookPoint,
    sae: SAE,
) -> Tensor:
    """
    Steers the model by returning a modified activations tensor, with some multiple of the steering vector added to all
    sequence positions.
    """
    return sae(activations)


# Load files

In [5]:
import json

# Read from advbench.json file
with open('../dataset/processed/advbench.json', 'r') as file:
    advbench_data = json.load(file)

len(advbench_data)

# Read from advbench.json file
with open('../dataset/processed/alpaca.json', 'r') as file:
    alpaca_data = json.load(file)

print(len(alpaca_data))

31323


In [35]:
gemma2_sae(activation)

NameError: name 'activation' is not defined

In [ ]:
gemma2: HookedSAETransformer = HookedSAETransformer.from_pretrained("gemma-2-2b-it", device=device)

In [ ]:
layer = 5
sae_name = "gemma-scope-2b-pt-res-canonical"
sae_id = f"layer_{layer}/width_16k/canonical"

gemma2_sae, cfg_dict, sparsity = SAE.from_pretrained(
            release=sae_name,
            sae_id=sae_id,
            device=str(device),
)

Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:08<00:00,  4.10s/it]


Loaded pretrained model gemma-2-2b-it into HookedTransformer


In [13]:
filename = "../pipeline/runs/gemma-2-2b-it/direction.pt"
refusal_direction = enrichment_utils.load_tensor(filename)


/n/data2/hms/dbmi/sunyaev/lab/dlee/ai_safety/refusal_direction/notebooks/../scripts/enrichment_utils.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return tensor


In [6]:
prompt = advbench_data[0]["instruction"]

In [40]:
# Get activations on final token
refusal_layer = 15
hook_name = f'blocks.{refusal_layer}.hook_resid_pre'

gemma2_sae.use_error_term = True
_, original_cache = gemma2.run_with_cache_with_saes(
    prompt,
    saes=[gemma2_sae],
    stop_at_layer=refusal_layer + 1,
)
original_projection = perturbation.get_projection(refusal_direction, original_cache[hook_name])

gemma2_sae.use_error_term = False
_, original_cache = gemma2.run_with_cache_with_saes(
    prompt,
    saes=[gemma2_sae],
    stop_at_layer=refusal_layer + 1,
)
new_projection = perturbation.get_projection(refusal_direction, original_cache[hook_name])


In [41]:
_, original_cache = gemma2.run_with_cache_with_saes(
    prompt,
    saes=[gemma2_sae],
    stop_at_layer=refusal_layer + 1,
)
perturbed_final_resid_pre_store = t.zeros(original_cache[hook_name].shape, device=device)

_steering_hook = partial(
    sae_hook,
    sae=gemma2_sae,
)
def get_activation_perturbed(
    activation, hook
):
    '''
    Get the activation
    '''
    perturbed_final_resid_pre_store[:, :] = activation[:, :].detach()


intervention_logits = gemma2.run_with_hooks(
    prompt,
    fwd_hooks=[(gemma2_sae.cfg.hook_name, _steering_hook),
               (hook_name, get_activation_perturbed)],
    stop_at_layer=refusal_layer + 1,
)

steered_activation = perturbed_final_resid_pre_store.clone()

perturbation.get_projection(refusal_direction, steered_activation)

tensor([[-250.9713,   14.8129,    3.2937,   15.5270,    5.2173,    9.4614,
           30.4371,   27.7763,   15.1055,   13.5773,   10.7002,   16.7744,
           12.7759,   15.8280]], device='cuda:0', dtype=torch.float64)

In [16]:
original_projection

tensor([[222.7908,  36.8987,   7.7450,  14.1471,  12.1025,  16.3383,  34.9077,
          29.4183,  21.5174,  18.3529,  14.7136,  24.7200,  17.7741,  19.8504]],
       device='cuda:0', dtype=torch.float64)

In [17]:
new_projection

tensor([[-250.9713,   14.8129,    3.2937,   15.5270,    5.2173,    9.4614,
           30.4371,   27.7763,   15.1055,   13.5773,   10.7002,   16.7744,
           12.7759,   15.8280]], device='cuda:0', dtype=torch.float64)